In [19]:
from pathlib import Path

import polars as pl
import polars_h3 as plh3
import yaml


In [3]:
with open("../config.budapest.yaml") as fp:
    config = yaml.safe_load(fp)

In [23]:
DIR = f"{config['dirs']['output']}/{config['city']}"
RES = 8
HOME_THRESHOLD = 5
WORK_THRESHOLD = 5

In [20]:
Path(f"{DIR}/dynamic_commuting").mkdir(parents=True, exist_ok=True)

In [6]:
hw = pl.read_csv(f"{DIR}/homes/v2/home_work_h3_{RES}.csv")
hw.head(3)

device_id,h3_8_home,count_home,h3_8_work,count_work
i64,str,i64,str,i64
1,"""881e037857fffff""",53,"""881e037857fffff""",35
2,"""881e0378c1fffff""",4,"""881e037833fffff""",22
3,"""881e1d8091fffff""",245,"""881e1d8359fffff""",143


In [12]:
TO_DROP = ["number_of_pings", "latitude_std", "longitude_std"]

In [ ]:
columns = [
    "device_id",
    "interval_start",
    "latitude_mean",
    "longitude_mean",
    "interval_lenght",
]
result = pl.DataFrame()
for i in range(256):
    data = pl.read_parquet(f"{DIR}/stops/v2/{i:02x}.parquet", columns=columns)
    data = data.with_columns(
        pl.col.interval_start.dt.date().alias("day"),
        pl.col.interval_start.dt.time().alias("time"),
        pl.col.interval_start.dt.hour().alias("hour"),
        pl.col.interval_start.dt.weekday().alias("dow"),
        plh3.int_to_str(
            plh3.latlng_to_cell(pl.col("latitude_mean"), pl.col("longitude_mean"), RES)
        ).alias(f"h3_{RES}"),
    )
    data = data.with_columns(
        pl.when((pl.col.dow.is_between(1, 5)) & (pl.col.hour.is_between(9, 16)))
        .then(pl.lit("work"))
        .otherwise(
            pl.when(
                (pl.col.dow.is_between(1, 5)) & ((pl.col.hour < 8) | (pl.col.hour > 19))
                | (pl.col.dow > 5)
            )
            .then(pl.lit("home"))
            .otherwise(pl.lit("other"))
        )
        .alias("label")
    )
    data = data.join(hw, on="device_id")

    hn = (
        data.filter(
            (pl.col(f"h3_{RES}") == pl.col(f"h3_{RES}_home")) & (pl.col.label == "home")
        )
        .group_by(["device_id", "day"])
        .agg(pl.col(f"h3_{RES}").count().alias("home_count"))
        .sort(["device_id", "day"])
    )
    wn = (
        data.filter(
            (pl.col(f"h3_{RES}") == pl.col(f"h3_{RES}_work")) & (pl.col.label == "work")
        )
        .group_by(["device_id", "day"])
        .agg(pl.col(f"h3_{RES}").count().alias("work_count"))
        .sort(["device_id", "day"])
    )
    m = pl.concat(
        [
            hn.join(wn, on=["device_id", "day"], how="left")
            .fill_null(0)
            .sort(["device_id", "day"]),
            hn.join(wn, on=["device_id", "day"], how="right")
            .fill_null(0)
            .select("device_id", "day", "home_count", "work_count")
            .sort(["device_id", "day"]),
        ]
    ).sort(["device_id", "day"])
    result = pl.concat([result, m])
# runs 45s

In [22]:
result.write_csv(f"{DIR}/dynamic_commuting/home_work_stops_by_daily_{RES}.csv")

In [24]:
# result = pl.read_csv("../output/mpd/home_work_stops_by_daily.csv")
merged = (
    result.join(hw, on="device_id")
    .filter(
        (pl.col.count_home >= HOME_THRESHOLD) & (pl.col.count_work >= WORK_THRESHOLD)
    )
    .rename({"home_count": "stops_at_home", "work_count": "stops_at_work"})
)

In [25]:
merged

device_id,day,stops_at_home,stops_at_work,h3_8_home,count_home,h3_8_work,count_work
i64,date,u32,u32,str,i64,str,i64
1,2025-05-04,1,0,"""881e037857fffff""",53,"""881e037857fffff""",35
1,2025-05-05,1,2,"""881e037857fffff""",53,"""881e037857fffff""",35
1,2025-05-05,1,2,"""881e037857fffff""",53,"""881e037857fffff""",35
1,2025-05-07,0,2,"""881e037857fffff""",53,"""881e037857fffff""",35
1,2025-05-11,2,0,"""881e037857fffff""",53,"""881e037857fffff""",35
…,…,…,…,…,…,…,…
1652698,2024-01-17,0,1,"""881e030f45fffff""",18,"""881e0301a1fffff""",5
1652698,2024-02-14,3,0,"""881e030f45fffff""",18,"""881e0301a1fffff""",5
1652698,2024-02-15,4,0,"""881e030f45fffff""",18,"""881e0301a1fffff""",5


In [26]:
merged.write_csv(f"{DIR}/dynamic_commuting/number_of_stops_at_home_work_{RES}.csv")